##  Imports, load saved boundary-detection artifacts, load Dataset A

In [1]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = ARTIFACTS_DIR / "models"
PREPROCESSING_DIR = ARTIFACTS_DIR / "preprocessing"
METADATA_DIR = ARTIFACTS_DIR / "metadata"

# Load the FINALIZED boundary detector + its metadata, rather than
# hardcoding threshold/feature values here -- this guarantees this
# notebook can never silently drift out of sync with what was actually saved.
boundary_model = joblib.load(MODELS_DIR / "best_segmentation_model.pkl")
boundary_preprocessor = joblib.load(PREPROCESSING_DIR / "best_preprocessor.pkl")

with open(METADATA_DIR / "segmentation_model_metadata.json", "r", encoding="utf-8") as f:
    boundary_metadata = json.load(f)

BOUNDARY_EVENT_TYPES = set(boundary_metadata["boundary_event_types_used_as_candidates"])
FEATURE_COLUMNS = boundary_metadata["feature_columns"]
CATEGORICAL_FEATURES = boundary_metadata["categorical_features"]
NUMERIC_FEATURES = boundary_metadata["numeric_features"]
BOUNDARY_THRESHOLD = boundary_metadata["boundary_threshold"]
MIN_SEGMENT_DURATION_S = boundary_metadata["min_segment_duration_seconds"]

print("Loaded boundary model:", type(boundary_model).__name__)
print("Boundary threshold    :", BOUNDARY_THRESHOLD)
print("Min segment duration  :", MIN_SEGMENT_DURATION_S, "s")
print("Candidate event types :", sorted(BOUNDARY_EVENT_TYPES))


def _has_session_children(directory: Path) -> bool:
    try:
        return any(
            child.is_dir() and child.name.startswith("ses_")
            for child in directory.iterdir()
        )
    except (PermissionError, FileNotFoundError):
        return False


def find_dataset_sessions_root(search_roots, dataset_label, max_depth=3):
    tried = []
    seen = set()
    frontier = [(root, 0) for root in search_roots if root.exists()]
    while frontier:
        current, depth = frontier.pop(0)
        if current in seen:
            continue
        seen.add(current)
        tried.append(current)
        if _has_session_children(current):
            return current
        if depth < max_depth:
            try:
                for child in current.iterdir():
                    if child.is_dir():
                        frontier.append((child, depth + 1))
            except (PermissionError, FileNotFoundError):
                continue
    raise FileNotFoundError(
        f"Could not locate a sessions folder for {dataset_label}.\n"
        f"Checked {len(tried)} candidate directories, including:\n"
        + "\n".join(f"  - {p}" for p in tried[:15])
    )


DATASET_A_DIR = find_dataset_sessions_root(
    [PROJECT_ROOT / "notebooks" / "dataset_A", PROJECT_ROOT / "dataset_A"], "Dataset A"
)
print("\nDataset A sessions root:", DATASET_A_DIR)


def load_session_events(session_dir: Path) -> pd.DataFrame:
    """Same as segmentation_model_v2.ipynb, plus extracted_text capture
    (not needed for boundary detection, needed here for labeling features)."""
    event_files = sorted(session_dir.rglob("events.jsonl"))
    if not event_files:
        raise FileNotFoundError(f"No events.jsonl files found under {session_dir}")

    records = []
    for event_file in event_files:
        chunk_dir_name = event_file.parent.name
        with open(event_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    event = json.loads(line)
                except json.JSONDecodeError:
                    continue

                context = event.get("context") or {}
                active_app = context.get("active_app") or {}
                correlation = event.get("correlation") or {}

                records.append({
                    "session_id": session_dir.name,
                    "event_id": event.get("event_id"),
                    "timestamp_iso": event.get("timestamp_iso"),
                    "layer": event.get("layer"),
                    "event_type": event.get("event_type"),
                    "active_app_name": active_app.get("app_name"),
                    "active_window_title": active_app.get("window_title"),
                    "extracted_text": context.get("extracted_text"),
                    "sequence_number": correlation.get("sequence_number"),
                    "source_chunk_dir": chunk_dir_name,
                })

    events_df = pd.DataFrame.from_records(records)
    events_df["timestamp"] = pd.to_datetime(
        events_df["timestamp_iso"].str.replace("Z", "+00:00", regex=False),
        utc=True, errors="coerce",
    )
    events_df = events_df.sort_values(
        by=["timestamp", "sequence_number"], kind="mergesort", na_position="last"
    ).reset_index(drop=True)
    return events_df


dataset_a_sessions = sorted([
    p for p in DATASET_A_DIR.iterdir() if p.is_dir() and p.name.startswith("ses_")
])
print("Sessions found:", len(dataset_a_sessions), "(expected 63)")

_start = time.perf_counter()
ALL_EVENTS_A = pd.concat([load_session_events(s) for s in dataset_a_sessions], ignore_index=True)
print(f"Loaded {len(ALL_EVENTS_A):,} events in {time.perf_counter() - _start:.1f}s (expected 162,768)")


Loaded boundary model: CatBoostClassifier
Boundary threshold    : 0.95
Min segment duration  : 15 s
Candidate event types : ['app_switch', 'browser_navigation', 'browser_tab_event', 'dialog_closed', 'dialog_opened', 'window_state_change', 'window_title_change']

Dataset A sessions root: c:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\notebooks\dataset_A\dataset_a_combined
Sessions found: 63 (expected 63)
Loaded 162,768 events in 14.2s (expected 162,768)


##  Load ground truth and reconstruct true segments (validation reference only)

In [2]:
def load_session_gt(session_dir: Path) -> list:
    gt_file = session_dir / "gt.jsonl"
    records = []
    with open(gt_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            record["session_id"] = session_dir.name
            records.append(record)
    records.sort(key=lambda r: r.get("ts_utc") or "")
    return records


_gt_records = []
for session_dir in dataset_a_sessions:
    _gt_records.extend(load_session_gt(session_dir))
ALL_GT_A = pd.DataFrame(_gt_records)


def reconstruct_gt_segments(session_gt: pd.DataFrame):
    session_gt = session_gt.sort_values("ts_utc").reset_index(drop=True)
    session_id = session_gt["session_id"].iloc[0]
    session_end_rows = session_gt[session_gt["event"] == "session_ended"]
    session_end_ts = (
        session_end_rows["ts_utc"].iloc[0] if not session_end_rows.empty
        else session_gt["ts_utc"].max()
    )
    relevant = session_gt[session_gt["event"].isin(
        ["process_started", "process_switched_out", "process_suspended", "process_resumed"]
    )]
    segments = []
    open_interval = {"active": False}

    def close_open_interval(end_ts, closed_by):
        if open_interval["active"]:
            segments.append({
                "session_id": session_id,
                "case_id": open_interval["case_id"],
                "process_code": open_interval["process_code"],
                "phase": open_interval["phase"],
                "start": open_interval["start_ts"],
                "end": end_ts,
            })
            open_interval["active"] = False

    for _, row in relevant.iterrows():
        event_type = row["event"]
        ts = row["ts_utc"]
        if event_type in ("process_started", "process_resumed"):
            close_open_interval(ts, f"implicit_{event_type}")
            open_interval.update({
                "active": True,
                "case_id": row.get("case_id"),
                "process_code": row.get("process_code") if pd.notna(row.get("process_code")) else row.get("current_process"),
                "phase": int(row["phase"]) if pd.notna(row.get("phase")) else 1,
                "start_ts": ts,
            })
        elif event_type in ("process_switched_out", "process_suspended"):
            close_open_interval(ts, event_type)

    close_open_interval(session_end_ts, "session_end_fallback")
    return segments


_all_segments = []
for session_id, group in ALL_GT_A.groupby("session_id"):
    _all_segments.extend(reconstruct_gt_segments(group))

GT_SEGMENTS_A = pd.DataFrame(_all_segments)
GT_SEGMENTS_A["start"] = pd.to_datetime(GT_SEGMENTS_A["start"], utc=True)
GT_SEGMENTS_A["end"] = pd.to_datetime(GT_SEGMENTS_A["end"], utc=True)

print("GT segments reconstructed:", len(GT_SEGMENTS_A), "(expected 2,009 -- validation reference only)")


GT segments reconstructed: 2009 (expected 2,009 -- validation reference only)


## Regenerate candidates + features, apply the saved boundary model

Identical feature engineering to `segmentation_model_v2.ipynb`, applied
across **all 63 sessions** this time (not just the held-out test split) --
we want a full set of predicted segments to build and validate the
labeling approach on.

In [3]:
CANDIDATES = ALL_EVENTS_A[ALL_EVENTS_A["event_type"].isin(BOUNDARY_EVENT_TYPES)].copy()
CANDIDATES = CANDIDATES.drop_duplicates(subset=["session_id", "timestamp"], keep="first")
CANDIDATES = CANDIDATES.sort_values(["session_id", "timestamp"]).reset_index(drop=True)

ALL_EVENTS_A = ALL_EVENTS_A.sort_values(["session_id", "timestamp"]).reset_index(drop=True)
ALL_EVENTS_A["time_since_prev_event_s"] = (
    ALL_EVENTS_A.groupby("session_id")["timestamp"].diff().dt.total_seconds()
)

CANDIDATES = CANDIDATES.merge(
    ALL_EVENTS_A[["event_id", "time_since_prev_event_s"]], on="event_id", how="left"
)
CANDIDATES = CANDIDATES.sort_values(["session_id", "timestamp"]).reset_index(drop=True)

CANDIDATES["gap_before_candidate_s"] = (
    CANDIDATES.groupby("session_id")["timestamp"].diff().dt.total_seconds()
)
session_start_map = ALL_EVENTS_A.groupby("session_id")["timestamp"].min()
CANDIDATES["time_since_session_start_s"] = (
    CANDIDATES["timestamp"] - CANDIDATES["session_id"].map(session_start_map)
).dt.total_seconds()

CANDIDATES["prev_app_name"] = CANDIDATES.groupby("session_id")["active_app_name"].shift(1)
CANDIDATES["prev_window_title"] = CANDIDATES.groupby("session_id")["active_window_title"].shift(1)
CANDIDATES["app_changed"] = (CANDIDATES["active_app_name"] != CANDIDATES["prev_app_name"]).astype(int)
CANDIDATES["title_changed"] = (CANDIDATES["active_window_title"] != CANDIDATES["prev_window_title"]).astype(int)

for event_type_name in sorted(BOUNDARY_EVENT_TYPES):
    CANDIDATES[f"is_{event_type_name}"] = (CANDIDATES["event_type"] == event_type_name).astype(int)


def count_events_in_window(session_events_ns, target_ns, window_ms, direction):
    if direction == "prior":
        lo = target_ns - window_ms * 1_000_000
        left = np.searchsorted(session_events_ns, lo, side="left")
        right = np.searchsorted(session_events_ns, target_ns, side="left")
    else:
        hi = target_ns + window_ms * 1_000_000
        left = np.searchsorted(session_events_ns, target_ns, side="right")
        right = np.searchsorted(session_events_ns, hi, side="right")
    return right - left


def add_window_counts(candidates_df, events_df, window_ms, direction, colname):
    counts = np.zeros(len(candidates_df), dtype=int)
    candidates_df = candidates_df.reset_index(drop=True)
    for session_id, cand_group in candidates_df.groupby("session_id"):
        session_events_ns = (
            events_df.loc[events_df["session_id"] == session_id, "timestamp"]
            .sort_values().values.astype("datetime64[ns]").astype("int64")
        )
        target_ns = cand_group["timestamp"].values.astype("datetime64[ns]").astype("int64")
        counts[cand_group.index.values] = count_events_in_window(session_events_ns, target_ns, window_ms, direction)
    candidates_df[colname] = counts
    return candidates_df


CANDIDATES = add_window_counts(CANDIDATES, ALL_EVENTS_A, 10000, "prior", "events_in_prior_10s")
CANDIDATES = add_window_counts(CANDIDATES, ALL_EVENTS_A, 10000, "next", "events_in_next_10s")

for numeric_column in NUMERIC_FEATURES:
    CANDIDATES[numeric_column] = CANDIDATES[numeric_column].fillna(0)

X_all = CANDIDATES[FEATURE_COLUMNS]
X_all_proc = boundary_preprocessor.transform(X_all)
CANDIDATES["boundary_probability"] = boundary_model.predict_proba(X_all_proc)[:, 1]
CANDIDATES["boundary_prediction"] = (CANDIDATES["boundary_probability"] >= BOUNDARY_THRESHOLD).astype(int)

print("Candidates:", len(CANDIDATES))
print("Predicted boundaries (pre-merge):", int(CANDIDATES["boundary_prediction"].sum()))


def merge_short_segments(boundary_timestamps, min_duration_s):
    if len(boundary_timestamps) == 0:
        return []
    kept = [boundary_timestamps[0]]
    for ts in boundary_timestamps[1:]:
        if (ts - kept[-1]).total_seconds() >= min_duration_s:
            kept.append(ts)
    return kept


def boundaries_to_segments(boundary_ts, session_id, session_end_ts):
    starts = list(boundary_ts)
    ends = starts[1:] + [session_end_ts]
    return pd.DataFrame({"session_id": session_id, "pred_start": starts, "pred_end": ends})


predicted_segments_list = []
for session_id, group in CANDIDATES[CANDIDATES["boundary_prediction"] == 1].groupby("session_id"):
    session_end_ts = ALL_EVENTS_A.loc[ALL_EVENTS_A["session_id"] == session_id, "timestamp"].max()
    boundary_ts = sorted(group["timestamp"].tolist())
    merged_ts = merge_short_segments(boundary_ts, MIN_SEGMENT_DURATION_S)
    if merged_ts:
        predicted_segments_list.append(boundaries_to_segments(merged_ts, session_id, session_end_ts))

PREDICTED_SEGMENTS_A = pd.concat(predicted_segments_list, ignore_index=True)
PREDICTED_SEGMENTS_A["pred_duration_s"] = (
    PREDICTED_SEGMENTS_A["pred_end"] - PREDICTED_SEGMENTS_A["pred_start"]
).dt.total_seconds()

print("Predicted segments across all of Dataset A:", len(PREDICTED_SEGMENTS_A),
      "(reference: 2,009 true segments)")


Candidates: 53109
Predicted boundaries (pre-merge): 1550
Predicted segments across all of Dataset A: 1494 (reference: 2,009 true segments)


## Match predicted segments to true process_code (validation reference only)

This mapping is used **only** in Cell 9 to check clustering quality --
never as an input feature to clustering itself.

In [4]:
def interval_iou(a_start_ns, a_end_ns, b_start_ns, b_end_ns):
    latest_start = max(a_start_ns, b_start_ns)
    earliest_end = min(a_end_ns, b_end_ns)
    intersection = max(0, earliest_end - latest_start)
    union = (a_end_ns - a_start_ns) + (b_end_ns - b_start_ns) - intersection
    return intersection / union if union > 0 else 0.0


def match_predicted_to_true(true_segments_df, pred_segments_df, iou_threshold=0.5):
    pred_segments_df = pred_segments_df.copy()
    pred_segments_df["matched_process_code"] = None
    pred_segments_df["matched_iou"] = 0.0

    for session_id in pred_segments_df["session_id"].unique():
        true_rows = true_segments_df[true_segments_df["session_id"] == session_id]
        pred_rows = pred_segments_df[pred_segments_df["session_id"] == session_id]
        used_pred_idx = set()
        for _, true_row in true_rows.iterrows():
            best_iou, best_idx = 0.0, None
            for pred_idx, pred_row in pred_rows.iterrows():
                if pred_idx in used_pred_idx:
                    continue
                score = interval_iou(
                    true_row["start"].value, true_row["end"].value,
                    pred_row["pred_start"].value, pred_row["pred_end"].value,
                )
                if score > best_iou:
                    best_iou, best_idx = score, pred_idx
            if best_idx is not None and best_iou >= iou_threshold:
                pred_segments_df.loc[best_idx, "matched_process_code"] = true_row["process_code"]
                pred_segments_df.loc[best_idx, "matched_iou"] = best_iou
                used_pred_idx.add(best_idx)

    return pred_segments_df


PREDICTED_SEGMENTS_A = match_predicted_to_true(GT_SEGMENTS_A, PREDICTED_SEGMENTS_A, iou_threshold=0.5)

n_matched = PREDICTED_SEGMENTS_A["matched_process_code"].notna().sum()
print(f"Predicted segments with a matched true process_code: {n_matched} / {len(PREDICTED_SEGMENTS_A)}")
print("(Unmatched segments are boundary false positives -- expected, given 87.2% test precision.)")


Predicted segments with a matched true process_code: 1331 / 1494
(Unmatched segments are boundary false positives -- expected, given 87.2% test precision.)


##  Check `extracted_text` coverage per segment before using it as a feature

The schema notes this field is populated on only ~4% of *events*. What
matters here is coverage per *segment* (does a typical 30-60s segment
contain at least one populated event), which could be meaningfully higher
than the flat 4% event-level rate. Decide whether to build text features
from this evidence, not from the schema note alone.

In [5]:
ALL_EVENTS_A_INDEXED = ALL_EVENTS_A.set_index("session_id")

def get_segment_events(session_id, start_ts, end_ts, events_by_session):
    if session_id not in events_by_session.groups:
        return pd.DataFrame(columns=ALL_EVENTS_A.columns)
    session_events = events_by_session.get_group(session_id)
    mask = (session_events["timestamp"] >= start_ts) & (session_events["timestamp"] < end_ts)
    return session_events.loc[mask]


events_by_session = ALL_EVENTS_A.groupby("session_id")

has_text_flags = []
text_event_counts = []
for _, seg in PREDICTED_SEGMENTS_A.iterrows():
    seg_events = get_segment_events(seg["session_id"], seg["pred_start"], seg["pred_end"], events_by_session)
    n_text = seg_events["extracted_text"].notna().sum()
    has_text_flags.append(n_text > 0)
    text_event_counts.append(n_text)

PREDICTED_SEGMENTS_A["has_any_extracted_text"] = has_text_flags
PREDICTED_SEGMENTS_A["n_extracted_text_events"] = text_event_counts

coverage = PREDICTED_SEGMENTS_A["has_any_extracted_text"].mean()
print(f"Fraction of predicted segments with at least one extracted_text event: {coverage:.1%}")
print("\n-> If this is reasonably high (say, above 40-50%), text-based keyword features")
print("   are worth building in Cell 6. If it is low, skip them -- sparse features that")
print("   only fire on a minority of segments add noise to clustering more than signal.")


Fraction of predicted segments with at least one extracted_text event: 99.3%

-> If this is reasonably high (say, above 40-50%), text-based keyword features
   are worth building in Cell 6. If it is low, skip them -- sparse features that
   only fire on a minority of segments add noise to clustering more than signal.


##  Build per-segment features (no ground truth involved)

App usage, event-type mix, timing, and activity-level features computed
directly from the raw events falling inside each predicted segment.
Whether text-keyword features are included depends on Cell 5's result --
this cell checks the coverage variable and includes them only if the
threshold set there is met.

In [6]:
TEXT_FEATURE_COVERAGE_THRESHOLD = 0.40  # decided from Cell 5's printed coverage number
USE_TEXT_FEATURES = coverage >= TEXT_FEATURE_COVERAGE_THRESHOLD
print("Using extracted_text features:", USE_TEXT_FEATURES)

EVENT_TYPE_FEATURE_LIST = [
    "keystroke", "mouse_click", "mouse_double_click", "mouse_scroll",
    "clipboard_change", "shortcut", "browser_click", "browser_form_input",
    "browser_navigation", "screenshot_smart",
]

feature_records = []
for _, seg in PREDICTED_SEGMENTS_A.iterrows():
    seg_events = get_segment_events(seg["session_id"], seg["pred_start"], seg["pred_end"], events_by_session)
    n_events = len(seg_events)

    record = {
        "session_id": seg["session_id"],
        "pred_start": seg["pred_start"],
        "pred_end": seg["pred_end"],
        "duration_s": seg["pred_duration_s"],
        "n_events": n_events,
        "n_distinct_apps": seg_events["active_app_name"].nunique(),
        "dominant_app": (
            seg_events["active_app_name"].mode().iloc[0]
            if n_events > 0 and not seg_events["active_app_name"].mode().empty
            else "unknown"
        ),
        "hour_of_day": seg["pred_start"].hour,
    }

    for event_type_name in EVENT_TYPE_FEATURE_LIST:
        record[f"frac_{event_type_name}"] = (
            (seg_events["event_type"] == event_type_name).sum() / n_events if n_events > 0 else 0.0
        )

    feature_records.append(record)

SEGMENT_FEATURES = pd.DataFrame(feature_records)

# Cyclical encoding for hour-of-day (23:00 and 00:00 are adjacent, not far apart)
SEGMENT_FEATURES["hour_sin"] = np.sin(2 * np.pi * SEGMENT_FEATURES["hour_of_day"] / 24)
SEGMENT_FEATURES["hour_cos"] = np.cos(2 * np.pi * SEGMENT_FEATURES["hour_of_day"] / 24)

# Collapse rare apps into "other" so one-hot encoding doesn't explode into
# one column per rarely-seen app name.
TOP_K_APPS = 12
top_apps = SEGMENT_FEATURES["dominant_app"].value_counts().head(TOP_K_APPS).index.tolist()
SEGMENT_FEATURES["dominant_app_grouped"] = SEGMENT_FEATURES["dominant_app"].where(
    SEGMENT_FEATURES["dominant_app"].isin(top_apps), "other"
)

print("Segment feature table shape:", SEGMENT_FEATURES.shape)
print("\nTop dominant apps:")
print(SEGMENT_FEATURES["dominant_app_grouped"].value_counts())


Using extracted_text features: True
Segment feature table shape: (1494, 21)

Top dominant apps:
dominant_app_grouped
Google Chrome           1338
Notepad                   93
Microsoft Word            43
Microsoft PowerPoint      14
Windows Explorer           4
claude                     1
ms-teams                   1
Name: count, dtype: int64


##  Encode features and sweep KMeans over a range of cluster counts (silhouette-selected)

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

LABEL_CATEGORICAL_FEATURES = ["dominant_app_grouped"]
LABEL_NUMERIC_FEATURES = (
    ["duration_s", "n_events", "n_distinct_apps", "hour_sin", "hour_cos"]
    + [f"frac_{name}" for name in EVENT_TYPE_FEATURE_LIST]
)

label_preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), LABEL_CATEGORICAL_FEATURES),
    ("numeric", StandardScaler(), LABEL_NUMERIC_FEATURES),
])

X_label_features = SEGMENT_FEATURES[LABEL_CATEGORICAL_FEATURES + LABEL_NUMERIC_FEATURES]
X_label_proc = label_preprocessor.fit_transform(X_label_features)
if hasattr(X_label_proc, "toarray"):
    X_label_proc = X_label_proc.toarray()

print("Encoded feature matrix shape:", X_label_proc.shape)

kmeans_results = []
for k in range(5, 21):
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
    cluster_labels = kmeans.fit_predict(X_label_proc)
    score = silhouette_score(X_label_proc, cluster_labels)
    kmeans_results.append({"k": k, "silhouette": score, "inertia": kmeans.inertia_})

kmeans_results_df = pd.DataFrame(kmeans_results).sort_values("silhouette", ascending=False)
print("\nKMeans silhouette by k (top 5, unsupervised selection -- no GT used):")
print(kmeans_results_df.head(5).to_string(index=False))

BEST_KMEANS_K = int(kmeans_results_df.iloc[0]["k"])
BEST_KMEANS_SILHOUETTE = float(kmeans_results_df.iloc[0]["silhouette"])
print(f"\nSelected k = {BEST_KMEANS_K} (silhouette = {BEST_KMEANS_SILHOUETTE:.3f})")


Encoded feature matrix shape: (1494, 22)

KMeans silhouette by k (top 5, unsupervised selection -- no GT used):
 k  silhouette      inertia
 8    0.179885 12003.665060
 9    0.178152 11347.266825
 7    0.169785 12677.279926
 6    0.169682 13720.324484
11    0.168650 10408.071232

Selected k = 8 (silhouette = 0.180)


##  Try HDBSCAN as a comparison (does not require choosing k up front)

Falls back gracefully and continues with KMeans only if HDBSCAN isn't
available in this environment.

In [8]:
HDBSCAN_AVAILABLE = True
try:
    from sklearn.cluster import HDBSCAN
except ImportError:
    HDBSCAN_AVAILABLE = False
    print("sklearn.cluster.HDBSCAN not available in this scikit-learn version.")
    print("Run: pip install --upgrade scikit-learn   (needs >= 1.3)")
    print("Continuing with KMeans only.")

hdbscan_results = []
if HDBSCAN_AVAILABLE:
    for min_cluster_size in [10, 15, 20, 30, 50]:
        hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size)
        cluster_labels = hdbscan_model.fit_predict(X_label_proc)
        non_noise_mask = cluster_labels != -1
        n_clusters_found = len(set(cluster_labels[non_noise_mask]))
        if n_clusters_found < 2 or non_noise_mask.sum() < 10:
            continue
        score = silhouette_score(X_label_proc[non_noise_mask], cluster_labels[non_noise_mask])
        hdbscan_results.append({
            "min_cluster_size": min_cluster_size,
            "n_clusters_found": n_clusters_found,
            "n_noise_points": int((~non_noise_mask).sum()),
            "silhouette_on_non_noise": score,
        })

    if hdbscan_results:
        hdbscan_results_df = pd.DataFrame(hdbscan_results).sort_values(
            "silhouette_on_non_noise", ascending=False
        )
        print("HDBSCAN results (unsupervised selection -- no GT used):")
        print(hdbscan_results_df.to_string(index=False))
    else:
        print("HDBSCAN did not produce a usable clustering (fewer than 2 clusters, or too")
        print("much noise) at any tested min_cluster_size. Proceeding with KMeans.")


HDBSCAN results (unsupervised selection -- no GT used):
 min_cluster_size  n_clusters_found  n_noise_points  silhouette_on_non_noise
               30                 2             375                 0.269338
               20                 3             351                 0.225725
               15                 3             295                 0.221560
               10                 4             223                 0.219964


## Pick the final clustering method, THEN validate against true process_code

Selection above used silhouette only. Ground truth is used here for the
first time, strictly as a validation check on the chosen method.

In [9]:
use_hdbscan = HDBSCAN_AVAILABLE and len(hdbscan_results) > 0 and (
    hdbscan_results_df.iloc[0]["silhouette_on_non_noise"] > BEST_KMEANS_SILHOUETTE
)

if use_hdbscan:
    chosen_min_cluster_size = int(hdbscan_results_df.iloc[0]["min_cluster_size"])
    final_clusterer = HDBSCAN(min_cluster_size=chosen_min_cluster_size)
    cluster_assignments = final_clusterer.fit_predict(X_label_proc)
    CHOSEN_METHOD = f"HDBSCAN(min_cluster_size={chosen_min_cluster_size})"
else:
    final_clusterer = KMeans(n_clusters=BEST_KMEANS_K, random_state=RANDOM_SEED, n_init=10)
    cluster_assignments = final_clusterer.fit_predict(X_label_proc)
    CHOSEN_METHOD = f"KMeans(k={BEST_KMEANS_K})"

SEGMENT_FEATURES["cluster_id"] = cluster_assignments
PREDICTED_SEGMENTS_A["cluster_id"] = cluster_assignments

print("Chosen clustering method:", CHOSEN_METHOD)
print("Cluster size distribution:")
print(SEGMENT_FEATURES["cluster_id"].value_counts().sort_index())

# ----------------------------------------------------------
# Validation against ground truth (labeled subset only -- unmatched
# predicted segments have no true process_code to compare against)
# ----------------------------------------------------------
from sklearn.metrics import adjusted_rand_score

labeled_mask = PREDICTED_SEGMENTS_A["matched_process_code"].notna() & (PREDICTED_SEGMENTS_A["cluster_id"] != -1)
labeled_clusters = PREDICTED_SEGMENTS_A.loc[labeled_mask, "cluster_id"]
labeled_true_codes = PREDICTED_SEGMENTS_A.loc[labeled_mask, "matched_process_code"]

print(f"\nLabeled, non-noise segments used for validation: {labeled_mask.sum()} / {len(PREDICTED_SEGMENTS_A)}")

crosstab = pd.crosstab(labeled_clusters, labeled_true_codes)
majority_counts = crosstab.max(axis=1)
purity = majority_counts.sum() / crosstab.values.sum()
ari = adjusted_rand_score(labeled_true_codes, labeled_clusters)

print(f"\nCluster purity (majority true process_code per cluster): {purity:.1%}")
print(f"Adjusted Rand Index vs true process_code               : {ari:.3f}")
print("\n(Purity near 100% / ARI near 1.0 = clusters strongly correspond to real processes.")
print(" Purity near 1/n_true_processes or ARI near 0 = feature representation isn't")
print(" capturing process identity well -- worth revisiting features before trusting")
print(" this approach on Dataset B.)")


Chosen clustering method: HDBSCAN(min_cluster_size=30)
Cluster size distribution:
cluster_id
-1     375
 0      39
 1    1080
Name: count, dtype: int64

Labeled, non-noise segments used for validation: 1045 / 1494

Cluster purity (majority true process_code per cluster): 10.3%
Adjusted Rand Index vs true process_code               : 0.001

(Purity near 100% / ARI near 1.0 = clusters strongly correspond to real processes.
 Purity near 1/n_true_processes or ARI near 0 = feature representation isn't
 capturing process identity well -- worth revisiting features before trusting
 this approach on Dataset B.)


##  Inspect cluster composition in full (crosstab)

In [10]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("Cluster x true-process_code crosstab (labeled, non-noise segments only):")
print(crosstab.to_string())


Cluster x true-process_code crosstab (labeled, non-noise segments only):
matched_process_code    A   B   C   D   E   F   G   H   I   J   K   L   M   N   O
cluster_id                                                                       
0                       1   3   2   2   1   3   1   7   0   1   5   3   0   2   6
1                     101  53  77  39  62  96  67  73  57  50  61  63  92  57  60


##  Assign readable labels and preview the segments.jsonl format

This is a dry run of the final output format on Dataset A -- the same
code path Dataset B will go through, just with Dataset A's predicted
segments and this notebook's clusterer (which, again, will NOT be reused
directly on B).

In [11]:
PREDICTED_SEGMENTS_A["label"] = PREDICTED_SEGMENTS_A["cluster_id"].apply(
    lambda cluster_id: f"process_cluster_{cluster_id}" if cluster_id != -1 else "unclustered_noise"
)

preview_columns = ["session_id", "pred_start", "pred_end", "label"]
preview_records = PREDICTED_SEGMENTS_A[preview_columns].head(10).copy()
preview_records["pred_start"] = preview_records["pred_start"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
preview_records["pred_end"] = preview_records["pred_end"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

print("Preview of segments.jsonl-format rows (Dataset A dry run, first 10):")
for _, row in preview_records.iterrows():
    print(json.dumps({
        "session_id": row["session_id"],
        "start": row["pred_start"],
        "end": row["pred_end"],
        "label": row["label"],
    }, ensure_ascii=False))

print("\nTotal predicted segments (Dataset A):", len(PREDICTED_SEGMENTS_A))
print("Distinct labels used:", PREDICTED_SEGMENTS_A["label"].nunique())


Preview of segments.jsonl-format rows (Dataset A dry run, first 10):
{"session_id": "ses_20260630-121953-LAPTOP-R36BQBTE", "start": "2026-06-30T12:21:12Z", "end": "2026-06-30T12:21:41Z", "label": "process_cluster_0"}
{"session_id": "ses_20260630-121953-LAPTOP-R36BQBTE", "start": "2026-06-30T12:21:41Z", "end": "2026-06-30T12:23:46Z", "label": "unclustered_noise"}
{"session_id": "ses_20260630-121953-LAPTOP-R36BQBTE", "start": "2026-06-30T12:23:46Z", "end": "2026-06-30T12:26:02Z", "label": "unclustered_noise"}
{"session_id": "ses_20260630-121953-LAPTOP-R36BQBTE", "start": "2026-06-30T12:26:02Z", "end": "2026-06-30T12:26:32Z", "label": "process_cluster_0"}
{"session_id": "ses_20260630-121953-LAPTOP-R36BQBTE", "start": "2026-06-30T12:26:32Z", "end": "2026-06-30T12:26:55Z", "label": "process_cluster_0"}
{"session_id": "ses_20260630-121953-LAPTOP-R36BQBTE", "start": "2026-06-30T12:26:55Z", "end": "2026-06-30T12:27:38Z", "label": "unclustered_noise"}
{"session_id": "ses_20260630-121953-LAPTOP-

##  Save Dataset A labeling artifacts (clearly marked: not for reuse on B)

In [12]:
labeler_model_path = MODELS_DIR / "segment_labeler_dataset_a.pkl"
labeler_preprocessor_path = PREPROCESSING_DIR / "segment_labeler_preprocessor_dataset_a.pkl"
labeler_metadata_path = METADATA_DIR / "segment_labeler_dataset_a_metadata.json"

joblib.dump(final_clusterer, labeler_model_path)
joblib.dump(label_preprocessor, labeler_preprocessor_path)

labeler_metadata = {
    "dataset": "dataset_a",
    "reusable_on_dataset_b": False,
    "reason_not_reusable": (
        "Dataset B has a completely different set of business processes "
        "(different department, different applications). This clusterer's "
        "cluster centers/structure are specific to Dataset A's 15 processes "
        "and have no meaningful correspondence on Dataset B. Dataset B "
        "requires refitting this SAME APPROACH from scratch on its own segments."
    ),
    "chosen_method": CHOSEN_METHOD,
    "categorical_features": LABEL_CATEGORICAL_FEATURES,
    "numeric_features": LABEL_NUMERIC_FEATURES,
    "top_k_apps_kept": TOP_K_APPS,
    "used_extracted_text_features": USE_TEXT_FEATURES,
    "extracted_text_segment_coverage": float(coverage),
    "n_predicted_segments": len(PREDICTED_SEGMENTS_A),
    "n_labeled_for_validation": int(labeled_mask.sum()),
    "validation_purity": float(purity),
    "validation_adjusted_rand_index": float(ari),
    "random_seed": RANDOM_SEED,
}

with open(labeler_metadata_path, "w", encoding="utf-8") as f:
    json.dump(labeler_metadata, f, indent=2, default=str)

print("Saved labeler model to :", labeler_model_path)
print("Saved preprocessor to  :", labeler_preprocessor_path)
print("Saved metadata to      :", labeler_metadata_path)


Saved labeler model to : c:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\models\segment_labeler_dataset_a.pkl
Saved preprocessor to  : c:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\preprocessing\segment_labeler_preprocessor_dataset_a.pkl
Saved metadata to      : c:\Users\Shambhavi Singh\OneDrive\Desktop\IBY_PROJECT\artifacts\metadata\segment_labeler_dataset_a_metadata.json


##  Verify saved artifacts by reloading

In [13]:
reloaded_labeler = joblib.load(labeler_model_path)
reloaded_label_preprocessor = joblib.load(labeler_preprocessor_path)

print("Reloaded labeler type      :", type(reloaded_labeler).__name__)
print("Reloaded label preprocessor:", type(reloaded_label_preprocessor).__name__)

with open(labeler_metadata_path, "r", encoding="utf-8") as f:
    print("\nSaved labeler metadata:")
    print(json.dumps(json.load(f), indent=2))


Reloaded labeler type      : HDBSCAN
Reloaded label preprocessor: ColumnTransformer

Saved labeler metadata:
{
  "dataset": "dataset_a",
  "reusable_on_dataset_b": false,
  "reason_not_reusable": "Dataset B has a completely different set of business processes (different department, different applications). This clusterer's cluster centers/structure are specific to Dataset A's 15 processes and have no meaningful correspondence on Dataset B. Dataset B requires refitting this SAME APPROACH from scratch on its own segments.",
  "chosen_method": "HDBSCAN(min_cluster_size=30)",
  "categorical_features": [
    "dominant_app_grouped"
  ],
  "numeric_features": [
    "duration_s",
    "n_events",
    "n_distinct_apps",
    "hour_sin",
    "hour_cos",
    "frac_keystroke",
    "frac_mouse_click",
    "frac_mouse_double_click",
    "frac_mouse_scroll",
    "frac_clipboard_change",
    "frac_shortcut",
    "frac_browser_click",
    "frac_browser_form_input",
    "frac_browser_navigation",
    "fra